- Conda environment: automlx251_p311_cpu_x86_64_v2
- Created Date: 05/03/2026
- By: Assaf Rabinowicz, EMEA Data Science Team

# Description

* In this notebook, we demonstrate how to register and deploy a machine learning model using the OCI SDK.
* While the ADS SDK (built on top of the OCI SDK) simplifies model registration and deployment, it can be limited for more advanced requirements.
* In this example, we register and deploy a model to a different compartment than the one where the Notebook Session resides and move deployment between compartments.
* We will still use ADS to create the model artifact.

# 1. Import Packages


In [ ]:
# third-party open-source packages
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_openml
import os
import requests
import zipfile

# Oracle packages
import automlx
from automlx import init
import oci
import ads
from ads.model import GenericModel

# 2. Data Import and Pre-Processing

In [ ]:
data = fetch_openml(name="adult", version=2, as_frame=True) # https://www.openml.org/search?type=data&sort=version&status=any&order=asc&exact_name=adult
df = data.frame

In [ ]:
df.drop(['fnlwgt'], axis=1,inplace=True) # dropping 'sampling weights' column for simplification

In [ ]:
df['class'] = (df['class'] == '>50K').astype(int)

# 3. Model Training

In [ ]:
X = df.drop('class', axis=1)
y = df['class']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.3) # 

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
init(engine='local')

In [ ]:
pipeline1 = automlx.Pipeline(task='classification',model_list=['LogisticRegression', 'RandomForestClassifier','XGBClassifier'],max_tuning_trials =10)
# model_list and max_tuning_trials were added to reduce fitting time. Removing them allows training a potentially better model.
# The automl pipeline has a rich api: https://docs.oracle.com/en-us/iaas/tools/automlx/latest/latest/automl.html 

In [ ]:
pipeline1.fit(X_train, y_train)

In [ ]:
y_train_pred = pipeline1.predict(X_train)
y_test_pred = pipeline1.predict(X_test)
print(y_test_pred[0:20])

# 4. Model Registration

## 4.1 Prepare the Artifacts (Serializiation) Using ADS

* Create the files required for deployment and pack them together.
* Besides the model, the following required files are generated automatically: `score.py`, `runtime.yaml`, `input_schema.json`, `output_schema.json`
* Optional info can be added, such as: `inference_conda_env`, `training_conda_env`
* The following frameworks have an automated prepare function: TensorFlow, PyTorch, scikit-learn, XGBoost, LightGBM, SparkPipelineModel, AutoMlx, transformers
* ADS takes you through the deployment process in a simple way

In [ ]:
ads.set_auth("resource_principal") # a signer for all ads operations, managed automatically

In [ ]:
automl_model = GenericModel(estimator=pipeline1, artifact_dir="<your_artifact_folder>")

In [ ]:
automl_model.summary_status() #ADS takes you thourgh the steps require for deployment.

In [ ]:
conda_env="automlx251_p311_cpu_x86_64_v2"
automl_model.prepare(inference_conda_env=conda_env,
                     training_conda_env=conda_env,
                     use_case_type=UseCaseType.BINARY_CLASSIFICATION,
                     X_sample=X_test,
                     force_overwrite=True)

In [ ]:
automl_model.verify(X_test.iloc[:20], auto_serialize_data=True)

In [ ]:
# In simple use cases, just continue with ADS for registration and deployment

# model_id = automl_model.save(display_name="Adults Income") # registration
# automl_model.deploy(display_name="Adults Income") # deployment

## 5.2 Register using OCI SDK

The steps are:
1. Authentication and creating a client
2. Zipping the your_artifact_folder
3. Register the model
4. Adding the zip file to the registered model
5. Additional metadata can be added.

In [ ]:
# Authentication
signer = oci.auth.signers.get_resource_principals_signer()
dsc = oci.data_science.DataScienceClient({}, signer=signer)
# config = oci.config.from_file()                           # Alternatively, you can also use config file
# dsc = oci.data_science.DataScienceClient(config)

In [ ]:
target_compartment_id = "<compartment_ocid>" # different that where the notebook resides
project_id ="<project_ocid_in_target_compartment>"

In [ ]:

def create_artifact_zip(source_dir, output_zip):
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(source_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, source_dir)
                zipf.write(file_path, arcname)

In [ ]:
create_artifact_zip("<your_artifact_folder_path>","<zip_path>")

In [ ]:
model_details = oci.data_science.models.CreateModelDetails(
    compartment_id=target_compartment_id, 
    project_id=project_id,
    display_name="<model_name>",
    description="Model prepared with ADS, registered via OCI SDK"
)

response = dsc.create_model(model_details)
model_id = response.data.id

In [ ]:
with open("<zip_path>", "rb") as artifact_file:
    artifact_bytes = artifact_file.read()   # reading into memory. Mandatory
    dsc.create_model_artifact(
            model_id=model_id,
            model_artifact=artifact_bytes, 
            content_disposition='attachment; filename="<zip_path>"'
        )

# 5. Deployment

In [ ]:
# Deployment setup
config_details = oci.data_science.models.SingleModelDeploymentConfigurationDetails(
    deployment_type="SINGLE_MODEL",
    model_configuration_details=oci.data_science.models.ModelConfigurationDetails(
        model_id=model_id,
        bandwidth_mbps=10,
        instance_configuration=oci.data_science.models.InstanceConfiguration(
            instance_shape_name="VM.Standard.E4.Flex",
            model_deployment_instance_shape_config_details=oci.data_science.models.ModelDeploymentInstanceShapeConfigDetails(
                ocpus=4.0,
                memory_in_gbs=16.0
            )
        ),
        scaling_policy=oci.data_science.models.FixedSizeScalingPolicy(
            policy_type="FIXED_SIZE",
            instance_count=1
        )
    )
)

In [ ]:
deployment_details = oci.data_science.models.CreateModelDeploymentDetails(
    display_name="New Deployment, custom 4",
    compartment_id=target_compartment_id,
    project_id=project_id,
    model_deployment_configuration_details=config_details  # ✅ Add here
)

response = dsc.create_model_deployment(deployment_details)
print(f"Deployment ID: {response.data.id}")

# Inference

In [ ]:
endpoint = '<endpoint>'

In [ ]:
body = {
    "data": '''[
        {
            "age": 37,
            "workclass": "Private",
            "education": "Bachelors",
            "education-num": 13,
            "marital-status": "Married-civ-spouse",
            "occupation": "Exec-managerial",
            "relationship": "Husband",
            "race": "White",
            "sex": "Male",
            "capital-gain": 50000,
            "capital-loss": 0,
            "hours-per-week": 40,
            "native-country": "United-States"
        }
    ]'''
}

requests.post(endpoint, json=body, auth=auth).json()

# 6 Moving Deployed Model Between Compartment

1. First, move the model resource.
2. Then move the model deployment using one of the following two approaches:
- Move the deployment directly using the change_model_deployment_compartment method (it can be done also without moving the model first).
- Deploy the model in the target compartment, by first extracting the deployment settings from the deployment in the source compartment, and then deploying the model in the target compartment using those settings.

In [ ]:
# Moving the model
model_change_details = oci.data_science.models.ChangeModelCompartmentDetails(compartment_id=target_compartment_id)
dsc.change_model_compartment(
            model_id=model_id,
            change_model_compartment_details=model_change_details
        )

# 6.1 Moving the deployment

In [ ]:
client.change_model_deployment_compartment(
    model_deployment_id="<model_deployment_id>",
    change_model_deployment_compartment_details=oci.data_science.models.ChangeModelDeploymentCompartmentDetails(
        compartment_id="<target_compartment_id>" # Make sure the model is activated. 
    )
)

# 6.2 Redeployment in the target compartment 

In [ ]:
old_deployment = dsc.get_model_deployment(model_deployment_id)
config_details = old_deployment.data.model_deployment_configuration_details

In [ ]:
new_deployment = oci.data_science.models.CreateModelDeploymentDetails(
    display_name=old_deployment.data.display_name,
    compartment_id=target_compartment_id,
    project_id='<project_id>',
    model_deployment_configuration_details=config_details
)
response = dsc.create_model_deployment(new_deployment)